# Bronze Layer: SQL Server Ingestion

This notebook extracts the Olist source tables from the local SQL Server `E_Commerce` database and writes an unchanged Parquet copy to `Bronze/staging`. The Bronze layer preserves source data for repeatable downstream processing.

## Inputs
- SQL Server running on `localhost:1433`
- Database: `E_Commerce`
- Source schema: `dbo`
- Microsoft SQL Server JDBC driver at `C:\spark_jar\mssql-jdbc-13.4.0.jre11.jar`

## Outputs
Parquet datasets are written below `Bronze/staging`: customers, category, orders, order_items, order_payments, order_reviews, geolocation, products, and sellers. The final cells compare source and staged row counts and inspect the customer schema.

Run this notebook before the Silver notebook. Update the local Java, JDBC driver, and SQL Server authentication settings for another environment.

In [2]:
import os

os.environ["JAVA_HOME"] = (
    r"C:\Program Files\Eclipse Adoptium\jdk-25.0.4.101-hotspot"
)

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# Add authentication DLL directory to PATH
os.environ["PATH"] = (
    r"C:\spark_jar;" + os.environ["PATH"]
)

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Olist_SQLServer_Ingestion")
    .master("local[*]")
    .config(
        "spark.jars",
        r"C:\spark_jar\mssql-jdbc-13.4.0.jre11.jar"
    )
    .getOrCreate()
)

print("Session started")
print("Spark version:", spark.version)

c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Session started
Spark version: 4.2.0


In [3]:
sqlserver_url = (
    "jdbc:sqlserver://localhost:1433;"
    "databaseName=E_Commerce;"
    "encrypt=true;"
    "trustServerCertificate=true;"
    "integratedSecurity=true;"
    "authenticationScheme=NativeAuthentication"
)

sqlserver_properties = {
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [15]:
def read_bronze(table_name):
    return spark.read.jdbc(
        url=sqlserver_url,
        table=f"dbo.{table_name}",
        properties=sqlserver_properties
    )

customers_df = read_bronze("customers")
category_df = read_bronze("category")
orders_df = read_bronze("orders")
order_items_df = read_bronze("order_items")
payments_df = read_bronze("order_payments")
sellers_df = read_bronze("sellers")
reviews_df = read_bronze("order_reviews")
geolocation_df = read_bronze("geolocation")
products_df = read_bronze("products")

In [16]:
def write_staging(df, table_name):
    df.write.mode("overwrite").parquet(f"staging/{table_name}")
    print(f"staging/{table_name} written")


write_staging(customers_df, "customers")
write_staging(category_df, "category")
write_staging(orders_df, "orders")
write_staging(order_items_df, "order_items")
write_staging(payments_df, "order_payments")
write_staging(sellers_df, "sellers")
write_staging(reviews_df, "order_reviews")
write_staging(geolocation_df, "geolocation")
write_staging(products_df, "products")

staging/customers written
staging/category written
staging/orders written
staging/order_items written
staging/order_payments written
staging/sellers written
staging/order_reviews written
staging/geolocation written
staging/products written


In [18]:
tables_to_check = {
    "customers": customers_df,
    "category": category_df,
    "orders": orders_df,
    "order_items": order_items_df,
    "order_payments": payments_df,
    "sellers": sellers_df,
    "order_reviews": reviews_df,
    "geolocation": geolocation_df,
    "products": products_df
}

for name, source_df in tables_to_check.items():
    staged_df = spark.read.parquet(f"staging/{name}")
    match = source_df.count() == staged_df.count()
    print(f"{name} — source: {source_df.count()}, staged: {staged_df.count()}, match: {match}")

customers — source: 99441, staged: 99441, match: True
category — source: 71, staged: 71, match: True
orders — source: 99441, staged: 99441, match: True
order_items — source: 112650, staged: 112650, match: True
order_payments — source: 103886, staged: 103886, match: True
sellers — source: 3095, staged: 3095, match: True
order_reviews — source: 99224, staged: 99224, match: True
geolocation — source: 1000163, staged: 1000163, match: True
products — source: 32951, staged: 32951, match: True


In [9]:
customers_df = (
    spark.read
    .jdbc(
        url=sqlserver_url,
        table="dbo.customers",
        properties=sqlserver_properties
    )
)

customers_df.show(5, truncate=False)

+--------------------------------+--------------------------------+------------------------+-------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city|customer_state|
+--------------------------------+--------------------------------+------------------------+-------------+--------------+
|00012a2ce6f8dcda20d059ce98491703|248ffe10d632bebe4f7267f1f44844c9|6273                    |osasco       |SP            |
|000161a058600d5901f007fab4c27140|b0015e09bb4b6e47c52844fab5fb6638|35550                   |itapecerica  |MG            |
|0001fd6190edaaf884bcaf3d49edf079|94b11d37cd61cb2994a194d11f89682b|29830                   |nova venecia |ES            |
|0002414f95344307404f0ace7a26f1d5|4893ad4ea28b2c5b3ddf4e82e79db9e6|39664                   |mendonca     |MG            |
|000379cdec625522490c315e70c7a9fb|0b83f73b19c2019e182fd552c048a22c|4841                    |sao paulo    |SP            |
+-----------------------

In [5]:
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

